# Deep Q-Network (DQN)

* Full Name: REDACTED
* Student Number: REDACTED

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque, namedtuple
import random
import matplotlib.pyplot as plt
import gymnasium as gym

In [ ]:
# MP4 Libraries
from IPython.display import HTML
from base64 import b64encode
import cv2
import numpy as np

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Helper Functions for Video

Function to convert MP4 to displayable HTML5 video and display

In [ ]:
def show_video(video_path, width=480):
    with open(video_path, "rb") as f:
        video_data = f.read()
    encoded = b64encode(video_data).decode("utf-8")
    html = f'''
        <video width="{width}" controls>
            <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
    '''
    return HTML(html)

def save_agent_video(env, agent, filename="agent_demo.mp4"):
    frames = []
    state, _ = env.reset()
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        frames.append(env.render())
        state = next_state

    height, width, layers = frames[0].shape
    out = cv2.VideoWriter(
        filename, cv2.VideoWriter_fourcc(*"mp4v"), 30, (width, height)
    )
    for f in frames:
        out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    out.release()


### Checkout the Environment


In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")

print("Action space:", env.action_space)
print("Action space dim:", env.action_space.n)

print("Observation space:", env.observation_space)
print("Observation space dim:", env.observation_space.shape[0])

class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space

    def select_action(self, state):
        return self.action_space.sample()


random_agent = RandomAgent(env.action_space)

video_path = "random_lunar_lander_rollout.mp4"
save_agent_video(env, random_agent, filename=video_path)

show_video(video_path)

### Replay Buffer
In DQN, the agent learns from past experiences.  
This buffer stores transitions in the form of (state, action, reward, next_state, done).

+ Your task: complete the `push` and `sample` methods.

- `push` should add a new Transition into the buffer.
- `sample` should randomly return a batch of transitions  
  (this helps break correlations and stabilize learning).

In [ ]:
Transition = namedtuple('Transition', 
                        ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    def __init__(self, capacity: int = 100000):
        self.memory = deque(maxlen=capacity)
    
    def push(self, *args):
        self.memory.append(Transition(*args))
    
    def sample(self, batch_size: int) -> Transition:
        transitions = random.sample(self.memory, batch_size)

        batch = Transition(*zip(*transitions))
        return batch
    
    def __len__(self):
        return len(self.memory)

### Q-Network

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super(QNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, x):
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32, device=device)
        return self.network(x)

###  Hyperparameters of the DQN Agent
These values control exploration, learning stability, and update frequency.
Feel free to experiment — small changes can significantly affect performance.

- `gamma`: discount factor  
- `epsilon`: initial exploration rate  
- `epsilon_decay`: how fast the agent shifts from exploration to exploitation  
- `batch_size`: number of samples from ReplayBuffer for each update  
- `target_update_freq`: how often the target network is updated


### Action Selection (Epsilon-Greedy)

The agent chooses between:
- exploration (random action) with probability epsilon  
- exploitation (best predicted action) otherwise


### Update Step of DQN

This is the heart of the algorithm:
1. Sample a batch from ReplayBuffer  
2. Compute predicted Q-values (current network)  
3. Compute target Q-values (target network)  
4. Calculate loss  
5. Optimize network weights  
6. Periodically update the target network

+ Your task: fill in the missing steps and complete the TODOs.
Follow the comments carefully — they guide the flow of DQN training.


In [ ]:
class DQNAgent:
    """
    Simple DQN Agent
    
    Key components:
    - Q-Network: approximates Q(s,a)
    - Target Network: provides stable targets for learning
    - Replay Buffer: stores past experiences
    - Epsilon-greedy: balances exploration/exploitation
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        self.gamma = 0.99
        self.epsilon = 1.0 
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.batch_size = 64
        self.target_update_freq = 1000
        self.action_dim = action_dim
        
        self.q_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=1e-3)
        self.memory = ReplayBuffer(100000)
        self.steps = 0
    
    def select_action(self, state, training=True):        
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            q_values = self.q_network(state)
        
        return torch.argmax(q_values).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):        
        if len(self.memory) < self.batch_size:
            return None
        
        batch = self.memory.sample(self.batch_size)
        
        states = torch.tensor(np.array(batch.state), dtype=torch.float32, device=device)
        actions = torch.tensor(batch.action, dtype=torch.long, device=device).unsqueeze(1)
        rewards = torch.tensor(batch.reward, dtype=torch.float32, device=device).unsqueeze(1)
        next_states = torch.tensor(np.array(batch.next_state), dtype=torch.float32, device=device)
        dones = torch.tensor(batch.done, dtype=torch.float32, device=device).unsqueeze(1)
        
        
        current_q = self.q_network(states).gather(1, actions)
        
        with torch.no_grad():
            next_q = self.target_network(next_states).max(1, keepdim=True)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        loss = nn.MSELoss()(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

### Training Loop

This function orchestrates the entire training process:
- Runs episodes
- Tracks scores and losses
- Updates the agent at each timestep
- Prints progress

+ Your task: you only need to complete the parts marked with TODO.
After that, run the notebook to train your agent!


In [ ]:
def train_dqn(env, agent, num_episodes=500, max_steps=1000):
    scores = []
    losses = []
    
    for episode in range(num_episodes):
        state, _ = env.reset(seed=SEED + episode)
        score = 0
        
        for step in range(max_steps):
            # Select action
            action = agent.select_action(state, training=True)
            
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store and update
            agent.store_transition(state, action, reward, next_state, done)
            loss = agent.update()
            
            score += reward
            state = next_state
            
            if done:
                break
        
        scores.append(score)
        # If no update occurred (e.g., buffer not full yet), store 0
        losses.append(loss if loss is not None else 0)
        
        if (episode + 1) % 50 == 0:
            avg = np.mean(scores[-50:])
            print(f"Episode {episode+1}/{num_episodes} | "
                  f"Score: {score:.1f} | Avg: {avg:.1f} | ε: {agent.epsilon:.3f}")
    
    return scores, losses


In [ ]:
print("="*60)
print("TRAINING SIMPLE DQN ON LUNAR LANDER")
print("="*60)

env = gym.make('LunarLander-v3', render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dim: {state_dim}, Action dim: {action_dim}\n")

# Create agent
dqn_agent = DQNAgent(state_dim, action_dim)

# Train
dqn_scores, dqn_losses = train_dqn(env, dqn_agent, num_episodes=500, max_steps=1000)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_dqn = "dqn.mp4"
save_agent_video(env, dqn_agent, filename=video_dqn)

axes[0].plot(dqn_scores)
axes[0].plot(np.convolve(dqn_scores, np.ones(50)/50, mode='valid'),
             'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dqn_losses)
axes[1].set_title('DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDQN Final Avg Score (last 50): {np.mean(dqn_scores[-50:]):.1f}")

show_video(video_dqn, width=600)


# Double DQN (DDQN)

### The Problem We're Solving
Regular DQN uses the **same network** to pick the best action AND to evaluate it.
This causes **overestimation** — the agent thinks actions are better than they really are.

Think of it like grading your own test: you're probably too nice to yourself!

### The Solution: Split the Jobs
Double DQN uses **two networks**:
- **Online Network** → picks the best action (the student)
- **Target Network** → evaluates that action's value (the teacher)


### Hyperparameters
Same as DQN — no new hyperparameters needed!
- `gamma`: discount factor
- `epsilon`: exploration rate
- `epsilon_decay`: how fast we stop exploring
- `batch_size`: samples per update
- `target_update_freq`: how often to sync networks

### The Double DQN Update Step
This is where the magic happens:

1. Sample a batch from ReplayBuffer
2. Get current Q-values from **online network**
3. Find **best action** using online network  $\to a_\text{best} = \argmax Q_\text{online}(s')$
4. Get Q-values for those actions from **target network**  $\to Q_\text{target}(s', a_\text{best})$
5. Compute TD target: $y = r + \gamma \times Q_\text{target}(s', a_\text{best})$
6. Compute loss between online predictions and target
7. Optimize online network
8. Periodically update target network

In [ ]:
class DDQNAgent(DQNAgent):
    """
    Double DQN extends DQN to reduce overestimation.
    
    The problem with standard DQN:
    - Uses max() which tends to overestimate Q-values
    
    Double DQN solution:
    - Use online network to SELECT the action
    - Use target network to EVALUATE the action
    
    This decouples selection from evaluation.
    """
    
    def update(self):
        """Update using Double DQN algorithm"""
        if len(self.memory) < self.batch_size:
            return None
        
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)
        
        # DOUBLE DQN Update
        
        # Current Q values (same as DQN)
        # q_network outputs Q(s, a) for all actions; we gather the ones for the taken actions
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            # Step 1: Use ONLINE network to select best action for next_states
            # argmax_a Q_online(s', a)
            next_actions = self.q_network(next_states).argmax(dim=1)
            
            # Step 2: Use TARGET network to evaluate that action
            # Q_target(s', a_best)
            next_q = self.target_network(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            
            # Step 3: Compute target
            # y = r + gamma * Q_target(s', a_best) * (1 - done)
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Compute loss and update (same as DQN)
        loss = nn.MSELoss()(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update target network periodically
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()


In [ ]:
print("="*60)
print("TRAINING DOUBLE DQN ON LUNAR LANDER")
print("="*60)

# Reset seeds for fair comparison
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env2 = gym.make('LunarLander-v3', render_mode="rgb_array")
state_dim2 = env2.observation_space.shape[0]
action_dim2 = env2.action_space.n

# Create DDQN agent
ddqn_agent = DDQNAgent(state_dim2, action_dim2)

# Train
ddqn_scores, ddqn_losses = train_dqn(env2, ddqn_agent, num_episodes=500, max_steps=1000)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_ddqn = "ddqn.mp4"
save_agent_video(env2, ddqn_agent, filename=video_ddqn)

axes[0].plot(ddqn_scores, alpha=0.5, label='DDQN')
axes[0].plot(np.convolve(ddqn_scores, np.ones(50)/50, mode='valid'),
             'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('Double DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ddqn_losses)
axes[1].set_title('Double DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDDQN Final Avg Score (last 50): {np.mean(ddqn_scores[-50:]):.1f}")

show_video(video_ddqn, width=600)


# Dueling DQN

### DuelingQNetwork — Splitting Value and Advantage

Instead of estimating $Q(s,a)$ directly, we break it into two parts:


- **$V(s)$** = How good is it to be in this state? (regardless of action)
- **$A(s,a)$** = How much better is action $a$ compared to other actions?

### Why Is This Better?
Imagine you're watching a movie:
- Some scenes are exciting regardless of what happens → **$V(s)$** is high
- The specific action matters more in tense scenes → **$A(s,a)$** carries more weight

By separating them, the network can learn **$V(s)$** directly without averaging across all actions.



### The Formula
We don't use $Q = V + A$ directly. Instead:

$$ Q(s,a) = V(s) + A(s,a) - mean(A(s,:))$$

In [ ]:
# =============================================================================
# Dueling DQN
# Architecture that separates value and advantage estimation
# =============================================================================
class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super(DuelingQNetwork, self).__init__()
        self.action_dim = action_dim
        
        # Shared feature extractor
        self.features = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        # Value stream: V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)   # Single value
        )
        
        # Advantage stream: A(s,a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
    
    def forward(self, x):
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32, device=device)
        
        features = self.features(x)
        
        # Get value and advantage
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        
        # Combine: Q(s,a) = V(s) + A(s,a) - mean(A(s,:))
        q_values = value + advantage - advantage.mean(dim=1, keepdim=True)
        
        return q_values

In [ ]:
class DuelingDQNAgent:
    def __init__(self, state_dim: int, action_dim: int):
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.batch_size = 64
        self.target_update_freq = 1000
        self.action_dim = action_dim
        
        self.q_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=1e-3)
        self.memory = ReplayBuffer(capacity=100000)
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            q_values = self.q_network(state)
        
        return torch.argmax(q_values).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        if len(self.memory) < self.batch_size:
            return None
        
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)
        
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            # Double DQN logic (important!)
            next_actions = self.q_network(next_states).argmax(dim=1)
            
            next_q = self.target_network(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Loss
        loss = nn.MSELoss()(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Target update
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Epsilon decay
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING DUELING DQN ON LUNAR LANDER")
print("="*60)

env3 = gym.make('LunarLander-v3', render_mode="rgb_array")
state_dim = env3.observation_space.shape[0]
action_dim = env3.action_space.n

dueling_agent = DuelingDQNAgent(state_dim, action_dim)

dueling_scores, dueling_losses = train_dqn(
    env3, dueling_agent, num_episodes=500, max_steps=1000
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

video_duelingdqn = "duelingdqn.mp4"
save_agent_video(env3, dueling_agent, filename=video_duelingdqn)

axes[0].plot(dueling_scores)
axes[0].plot(
    np.convolve(dueling_scores, np.ones(50)/50, mode='valid'),
    'r-', linewidth=2, label='Moving Avg'
)
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('Dueling DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dueling_losses)
axes[1].set_title('Dueling DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDueling DQN Final Avg Score (last 50): {np.mean(dueling_scores[-50:]):.1f}")

show_video(video_duelingdqn, width=600)

# Prioritized Experience Replay (PER)

### SumTree — The Priority Organizer

### Why Do We Need It?
Regular replay buffer picks experiences **randomly**.
But some experiences teach us more than others!

Experiences with **high TD error** (big surprise) are more valuable.
We want to pick those more often.

### How Does SumTree Work?
Think of it like a **binary tree** where:
- Each **leaf** = one experience with a priority score
- Each **internal node** = sum of its children


In [ ]:
class SumTree:
    def __init__(self, capacity: int):
        self.capacity = capacity
        
        self.tree = np.zeros(2 * capacity - 1)
        
        self.data = np.zeros(capacity, dtype=object)
        
        self.write_idx = 0
        self.size = 0
    
    def add(self, priority: float, transition: Transition):
        idx = self.write_idx + self.capacity - 1
        
        self.data[self.write_idx] = transition
        self.update(idx, priority)
        
        self.write_idx = (self.write_idx + 1) % self.capacity
        
        self.size = min(self.size + 1, self.capacity)
    
    def update(self, idx: int, priority: float):
        change = priority - self.tree[idx]
        self.tree[idx] = priority
        
        while idx != 0:
            idx = (idx - 1) // 2
            self.tree[idx] += change
    
    def sample(self, batch_size: int) -> list:
        batch = []
        
        total_priority = self.tree[0]
        segment = total_priority / batch_size
        
        for i in range(batch_size):
            a = segment * i
            b = segment * (i + 1)
            
            r = random.uniform(a, b)
            
            idx = self._find_leaf(0, r)
            
            data_idx = idx - self.capacity + 1
            batch.append(self.data[data_idx])
        
        return batch
    
    def _find_leaf(self, idx: int, r: float) -> int:
        while True:
            left = 2 * idx + 1
            right = left + 1
            
            if left >= len(self.tree):
                return idx
            
            if r <= self.tree[left]:
                idx = left
            else:
                r -= self.tree[left]
                idx = right
    
    def __len__(self):
        return self.size

### The Problem with Uniform Sampling
Regular buffer treats all experiences equally.
But if you already know something well, reviewing it is a waste of time!

### The Solution: Priority-Based Sampling
We sample experiences with probability proportional to their TD error:

+ ${P}(\text{experience}) \propto  |\text{TD error}|^\alpha $

- $\alpha = 0 \to$ uniform sampling (like regular buffer)
- $\alpha = 1 \to$ full prioritization (only important experiences)

### Importance Sampling (IS) Weights
Prioritization changes the distribution → introduces **bias**.
We fix this with IS weights:
+ $w = (N × \mathbb{P}(i))^{-\beta}$

- $\beta = 0 \to$ no correction (fast but biased)
- $\beta = 1 \to$ full correction (unbiased but higher variance)


In [ ]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int = 100000, alpha: float = 0.6, beta: float = 0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        
        self.beta_increment = 1e-3
        
        self.epsilon = 1e-5
        
        self.tree = SumTree(capacity)
        self.max_priority = 1.0
    
    def push(self, *args):
        transition = Transition(*args)
        
        priority = self.max_priority
        self.tree.add(priority, transition)
    
    def sample(self, batch_size: int):
        batch = []
        indices = []
        priorities = []
        
        total_priority = self.tree.tree[0]
        segment = total_priority / batch_size
        
        for i in range(batch_size):
            a = segment * i
            b = segment * (i + 1)
            
            r = random.uniform(a, b)
            
            idx = self.tree._find_leaf(0, r)
            data_idx = idx - self.tree.capacity + 1
            
            transition = self.tree.data[data_idx]
            
            batch.append(transition)
            indices.append(idx)
            priorities.append(self.tree.tree[idx])
        
        priorities = np.array(priorities)
        
        probs = priorities / total_priority
        
        weights = (self.tree.size * probs) ** (-self.beta)
        weights /= weights.max()
        
        self.beta = min(1.0, self.beta + self.beta_increment)
        
        batch = Transition(*zip(*batch))
        
        states = np.array(batch.state)
        actions = np.array(batch.action)
        rewards = np.array(batch.reward)
        next_states = np.array(batch.next_state)
        dones = np.array(batch.done)
        
        return states, actions, rewards, next_states, dones, weights, indices
    
    def update_priorities(self, indices: list, td_errors: np.ndarray):
        """Update priorities using TD error"""
        for idx, error in zip(indices, td_errors):
            priority = (abs(error) + self.epsilon) ** self.alpha
            self.tree.update(idx, priority)
            self.max_priority = max(self.max_priority, priority)
    
    def __len__(self):
        return len(self.tree)

In [ ]:
class PERDQNAgent:
    def __init__(self, state_dim: int, action_dim: int):
        # Standard DQN hyperparameters
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.batch_size = 64
        self.target_update_freq = 1000
        self.action_dim = action_dim
        
        # PER parameters
        self.alpha = 0.6
        self.beta = 0.4
        self.beta_increment = 1e-3
        
        # Networks
        self.q_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=1e-3)
        
        # Prioritized Replay Buffer
        self.memory = PrioritizedReplayBuffer(
            capacity=50000, alpha=self.alpha, beta=self.beta
        )
        
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            q_values = self.q_network(state)
        
        return torch.argmax(q_values).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        if len(self.memory) < self.batch_size:
            return None
        
        states, actions, rewards, next_states, dones, weights, indices = \
            self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_actions = self.q_network(next_states).argmax(dim=1)
            
            next_q = self.target_network(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        td_errors = (current_q - target_q).detach().cpu().numpy()
        
        weights_tensor = torch.FloatTensor(weights).to(device)
        
        loss = (weights_tensor * (current_q - target_q) ** 2).mean()
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.memory.update_priorities(indices, td_errors)
        
        self.beta = min(1.0, self.beta + self.beta_increment)
        self.memory.beta = self.beta
        
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING PER DQN ON LUNAR LANDER")
print("="*60)

# Reset seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env4 = gym.make('LunarLander-v3', render_mode="rgb_array")

state_dim = env4.observation_space.shape[0]
action_dim = env4.action_space.n

# Create agent
per_agent = PERDQNAgent(state_dim, action_dim)

# Train
per_scores, per_losses = train_dqn(
    env4, per_agent, num_episodes=500, max_steps=1000
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_perdqn = "perdqn.mp4"
save_agent_video(env4, per_agent, filename=video_perdqn)

axes[0].plot(per_scores)
axes[0].plot(
    np.convolve(per_scores, np.ones(50)/50, mode='valid'),
    'r-', linewidth=2, label='Moving Avg'
)
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('PER DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(per_losses)
axes[1].set_title('PER DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPER DQN Final Avg Score (last 50): {np.mean(per_scores[-50:]):.1f}")

show_video(video_perdqn, width=600)

# Half-Rainbow DQN (Double + Dueling + PER)

Rainbow DQN (Hessel et al., 2018) combined **6 different techniques**.
In this section we implement the 3 mentioned ones.


### Your Task
- Combine `DuelingQNetwork` architecture
- Use `PrioritizedReplayBuffer` for storage
- Implement Double DQN update logic
- Make sure priorities are updated after each learning step

In [ ]:
class CombinedDQNAgent:
    def __init__(self, state_dim: int, action_dim: int):
        # Hyperparameters
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.batch_size = 64
        self.target_update_freq = 1000
        self.action_dim = action_dim
        
        # PER parameters
        self.alpha = 0.6
        self.beta = 0.4
        self.beta_increment = 1e-3
        
        # Networks (Dueling)
        self.q_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=1e-3)
        
        # Prioritized Replay Buffer (FIXED capacity)
        self.memory = PrioritizedReplayBuffer(
            capacity=50000, alpha=self.alpha, beta=self.beta
        )
        
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            q_values = self.q_network(state)
        
        return torch.argmax(q_values).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        if len(self.memory) < self.batch_size:
            return None
        
        states, actions, rewards, next_states, dones, weights, indices = \
            self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_actions = self.q_network(next_states).argmax(dim=1)
            
            next_q = self.target_network(next_states).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)
            
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        td_errors = (current_q - target_q).detach().cpu().numpy()
        
        weights_tensor = torch.FloatTensor(weights).to(device)
        loss = (weights_tensor * (current_q - target_q) ** 2).mean()
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.memory.update_priorities(indices, td_errors)
        
        self.beta = min(1.0, self.beta + self.beta_increment)
        self.memory.beta = self.beta
        
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING HALF RAINBOW DQN (Double + Dueling + PER)")
print("="*60)

# Reset seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env5 = gym.make('LunarLander-v3', render_mode="rgb_array")

state_dim = env5.observation_space.shape[0]
action_dim = env5.action_space.n

combined_agent = CombinedDQNAgent(state_dim, action_dim)

combined_scores, combined_losses = train_dqn(
    env5, combined_agent, num_episodes=500, max_steps=1000
)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_halfrainbow = "halfrainbow.mp4"
save_agent_video(env5, combined_agent, filename=video_halfrainbow)

axes[0].plot(combined_scores)
axes[0].plot(
    np.convolve(combined_scores, np.ones(50)/50, mode='valid'),
    'r-', linewidth=2, label='Moving Avg'
)
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('Combined DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(combined_losses)
axes[1].set_title('Combined DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nCombined DQN Final Avg Score (last 50): {np.mean(combined_scores[-50:]):.1f}")

show_video(video_halfrainbow, width=600)

# Last Task: Final Comparison - All Methods

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

window = 50

# Calculate moving averages
dqn_ma = np.convolve(dqn_scores, np.ones(window)/window, mode='valid')
ddqn_ma = np.convolve(ddqn_scores, np.ones(window)/window, mode='valid')
dueling_ma = np.convolve(dueling_scores, np.ones(window)/window, mode='valid')
per_ma = np.convolve(per_scores, np.ones(window)/window, mode='valid')
combined_ma = np.convolve(combined_scores, np.ones(window)/window, mode='valid')

# ===================== Plot 1 =====================
# All moving averages
axes[0, 0].plot(dqn_ma, color='blue', linewidth=2, label='DQN')
axes[0, 0].plot(ddqn_ma, color='red', linewidth=2, label='Double DQN')
axes[0, 0].plot(dueling_ma, color='green', linewidth=2, label='Dueling DQN')
axes[0, 0].plot(per_ma, color='purple', linewidth=2, label='PER DQN')
axes[0, 0].plot(combined_ma, color='orange', linewidth=2, label='Combined')
axes[0, 0].axhline(y=200, color='black', linestyle='--', label='Target (200)')
axes[0, 0].set_title('All Methods - Moving Average (50)')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Score')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ===================== Plot 2 =====================
# Final scores bar chart
labels = ['DQN', 'Double\nDQN', 'Dueling\nDQN', 'PER\nDQN', 'Combined']
final_scores = [
    np.mean(dqn_scores[-50:]),
    np.mean(ddqn_scores[-50:]),
    np.mean(dueling_scores[-50:]),
    np.mean(per_scores[-50:]),
    np.mean(combined_scores[-50:])
]
colors = ['blue', 'red', 'green', 'purple', 'orange']

bars = axes[0, 1].bar(labels, final_scores, color=colors, alpha=0.7)
axes[0, 1].axhline(y=200, color='black', linestyle='--', label='Target')
axes[0, 1].set_title('Average Score (Last 50 Episodes)')
axes[0, 1].set_ylabel('Score')
axes[0, 1].legend()

for bar, score in zip(bars, final_scores):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, 
                    f'{score:.1f}', ha='center', fontsize=11, fontweight='bold')

# ===================== Plot 3 =====================
# Raw scores (shows variance / stability)
axes[1, 0].plot(dqn_scores, color='blue', alpha=0.3, label='DQN')
axes[1, 0].plot(ddqn_scores, color='red', alpha=0.3, label='Double DQN')
axes[1, 0].plot(dueling_scores, color='green', alpha=0.3, label='Dueling DQN')
axes[1, 0].plot(per_scores, color='purple', alpha=0.3, label='PER DQN')
axes[1, 0].plot(combined_scores, color='orange', alpha=0.3, label='Combined')
axes[1, 0].set_title('Raw Scores (Variance Comparison)')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ===================== Plot 4 =====================
# Sample efficiency (episodes to reach score 200)
def episodes_to_target(scores, target=200):
    for i, s in enumerate(scores):
        if s >= target:
            return i
    return len(scores)

eff_scores = [
    episodes_to_target(dqn_scores),
    episodes_to_target(ddqn_scores),
    episodes_to_target(dueling_scores),
    episodes_to_target(per_scores),
    episodes_to_target(combined_scores),
]

axes[1, 1].bar(labels, eff_scores, color=colors, alpha=0.7)
axes[1, 1].set_title('Sample Efficiency (Episodes to Reach 200)')
axes[1, 1].set_ylabel('Episodes')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ================= Summary =================
print("\n" + "="*70)
print("FINAL SUMMARY - ALL METHODS COMPARISON")
print("="*70)
print(f"{'Method':<20} {'Avg Score (last 50)':<25} {'Status':<15}")
print("-"*70)
print(f"{'DQN':<20} {np.mean(dqn_scores[-50:]):<25.1f} {'Basic':<15}")
print(f"{'Double DQN':<20} {np.mean(ddqn_scores[-50:]):<25.1f} {'Reduced overestimation':<15}")
print(f"{'Dueling DQN':<20} {np.mean(dueling_scores[-50:]):<25.1f} {'Better V(s) estimation':<15}")
print(f"{'PER DQN':<20} {np.mean(per_scores[-50:]):<25.1f} {'Better sampling':<15}")
print(f"{'Combined':<20} {np.mean(combined_scores[-50:]):<25.1f} {'All techniques':<15}")
print("="*70)
print(f"Target Score: 200 (solved)")